# Using a Next Generation Matrix to calculate $R_0$

For my model (ignoring the split into vaccinated/unvaccinated), the diseased classes are $E$, $I_a^S$, $I_a^A$, and $H_a$, where $a \in \{1,2,3 \}$ denotes the age group. Their equations are:

$$\dfrac{dE_a}{dt} =  \lambda_a \frac{S_a}{N_a} - \epsilon E_a$$ 

$$\dfrac{dI_a^S}{dt} = \epsilon d_a E_a - \gamma ' I_a^S$$

$$\dfrac{dI_a^A}{dt} = \epsilon (1-d_a) E_a - \gamma I_a^A$$

$$\dfrac{dI_a^R}{dt} = \gamma ' (1-h_a)I_a^S - \delta ' I_a^R$$

$$\dfrac{dH_a}{dt} = \gamma ' h_a I_a^S - \delta H_a$$

Where $\lambda_a$ denotes the force of infection, given by the equation:

$$ \lambda_a = \sum_{b=1}^3 \beta^{ba} (\tau I_b^A + I_b^S + I_b^R + \rho H_b)$$

We want to split these into two 5-dim vectors, where one ($\mathcal{F}$) contains the elements of each that contain the transmission rate, $\beta$, and the other, $\mathcal{V}$, contains the rest such that:

$$
\left( \dfrac{dE}{dt}, \dfrac{dI^S}{dt} , \dfrac{dI^A}{dt}, \dfrac{dI^R}{dt}, \dfrac{dH}{dt} \right)^{T} = \mathcal{F}(E,I^S, I^A, I^R, H) - \mathcal{V}(E,I^S, I^A, I^R, H)
$$

Therefore, for our equations,

$$
\mathcal{F} = 
\begin{pmatrix}
 \sum_{b=1}^3 \beta^{ba} (\tau I_b^A + I_b^S + I_b^R + \rho H_b)\frac{S}{N} \\
 0 \\
 0 \\
 0 \\
 0 \\
\end{pmatrix}
$$

and

$$
\mathcal{V} = 
\begin{pmatrix}
 \epsilon E \\
 -\epsilon d E + \gamma ' I^S \\
 -\epsilon (1-d) E + \gamma I^A \\
 - \gamma ' (1-h)I^S + \delta ' I^R \\
  -\gamma ' h I^S + \delta H\\
\end{pmatrix}
$$

Then, $F = Jacobian(\mathcal{F})$ and $V = Jacobian(\mathcal{V})$.

$R_0$ is given by the dominant (largest) eigenvalue of the 5x5 matrix $F V^{-1}$.

Here, the largest eigenvalue of that matrix - which is therefore $R_0$ - is:

$$ 
R_0 = \sum_{b=1}^3 \beta^{ba} \left( \frac{d}{\gamma '} +  \frac{\tau (1-d)}{\gamma} + \frac{d(1-h)}{\delta '} + \frac{dh \rho}{\delta}  \right)
$$

The below code calculates the matrices $F$ and $V$ for the given rates/probabilities, and then finds the largest eigenvalue of $FV^{-1}$.

In [34]:
import numpy as np
import json

f = open("params.json", "r")
contents = f.read()
f.close()


params = json.loads(contents)

    
youngAgeGrp = params['0-19']
adultAgeGrp = params['20-64']
elderlyAgeGrp = params['65+']

In [35]:
ageGrps = [youngAgeGrp, adultAgeGrp, elderlyAgeGrp]

betaSums = []
for age in ageGrps:
    betaSums.append(sum(age['beta1']))

tau = youngAgeGrp['tau1']
rho = youngAgeGrp['rho1']
epsilon = youngAgeGrp['epsilon1']
gamma = youngAgeGrp['gamma1']
gammaprime = youngAgeGrp['gamma1prime']
delta = youngAgeGrp['delta1']
deltaprime = youngAgeGrp['delta1prime']
d = [youngAgeGrp['d1'], adultAgeGrp['d1'], elderlyAgeGrp['d1'] ]
h = [youngAgeGrp['h1'], adultAgeGrp['h1'], elderlyAgeGrp['h1'] ]

R0_array = []
for i in range(len(betaSums)):
    F = [[0,betaSums[i], betaSums[i]*tau, betaSums[i], betaSums[i]*rho], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]] 

    V = [[epsilon, 0, 0, 0, 0], [-epsilon*d[i], gammaprime, 0, 0, 0], [-epsilon*(1-d[i]), 0, gamma, 0, 0], [0,-gammaprime*(1-h[i]), 0 , deltaprime, 0] , [0, -gammaprime*h[i], 0, 0, delta]]
    
    FV_inv = np.dot(F, np.linalg.inv(V))

    Evals = np.linalg.eig(FV_inv) #returns the four eigenvalues and eigenvectors

    R0_array.append( max(Evals[0]) ) #Evals[0] is an array of the eigenvalues

print("The basic reproductive number R0 for 0-19 year olds is:",R0_array[0])
print("R0 for 20-64 year olds is:", R0_array[1])
print("R0 for 65+ year olds is:", R0_array[2])

The basic reproductive number R0 for 0-19 year olds is: 2.5662474000000004
R0 for 20-64 year olds is: 3.05244
R0 for 65+ year olds is: 3.88992148
